# 🎯 Module 05: Structured Outputs and Validation

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 1 — Foundations  
> **Module:** 05 — Structured Outputs and Validation  
> **Planned Date:** July 10, 2026  
> **Level:** Beginner → Advanced  
> **Prerequisites:** Python, Modules 01–04  
> **Previous:** 04 — Prompt Engineering  
> **Next:** 06 — Embeddings and Semantic Search

---

LLMs generate **text**. Real applications need **structured, reliable data**.

In this module we'll learn how to turn model responses into **machine-readable** objects that our code can trust — using **JSON**, **schemas**, **Pydantic**, and **validation**.

We'll cover:

- Why **structured outputs** matter
- **JSON** fundamentals
- **Schemas** & required fields
- **JSON prompting** vs **JSON mode** vs **schema-constrained** output
- **Pydantic** models
- Validating **JSON strings** and **dicts**
- A **structured extraction pipeline**
- **Provider-enforced** structured output
- **Error handling** & recovery
- **Structured output vs tool calling**
- **Reliability, security, and best practices**
- A **practical lab**: Support Ticket Extractor

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain why LLM apps need **structured outputs**
2. Understand **JSON** objects, arrays, and types
3. Distinguish **JSON prompting** from **provider-enforced structured output**
4. Validate LLM responses using **Pydantic**
5. Handle **malformed JSON**, **missing fields**, **wrong types**
6. Apply **business rules** after schema validation
7. Design reusable **data extraction pipelines**
8. **Test** structured-output systems with edge cases
9. Safely use LLM-generated data in **backend applications**

---
## ⚙️ 0. Setup & Imports

We'll use **Pydantic** for validation. The notebook is safe to run **without** an API key.

In [ ]:
import os
import json
from typing import Literal

HAS_KEY   = bool(os.getenv('LLM_API_KEY'))
HAS_MODEL = bool(os.getenv('LLM_MODEL'))

print('✅ Setup ready.')
print(f'LLM_API_KEY present: {HAS_KEY}')
print(f'LLM_MODEL   present: {HAS_MODEL}')

# Check if pydantic is available
try:
    import pydantic
    print(f'Pydantic version: {pydantic.__version__}')
    HAS_PYDANTIC = True
except ImportError:
    print('Pydantic not installed. Run: pip install pydantic')
    HAS_PYDANTIC = False

if not (HAS_KEY and HAS_MODEL):
    print('\n⚠️ Live model calls disabled. Validation and parsing code still runs.')

---
## 🧠 1. Why Structured Outputs Matter

LLMs naturally generate **text**. Real applications often need **predictable data**.

### Example — customer support system

Instead of a paragraph, we need:

- Customer **issue**
- **Category**
- **Priority**
- **Sentiment**
- **Suggested response**

```json
{
  "issue": "Unable to reset password",
  "category": "account_access",
  "priority": "high",
  "sentiment": "frustrated",
  "suggested_response": "Please use the password reset link."
}
```

Python can **parse**, **validate**, **store**, and **display** those fields.

> ⚠️ **Valid JSON ≠ correct information.** You must validate **both structure and meaning**.

---
## 🧩 2. JSON Fundamentals

**JSON** = JavaScript Object Notation. A text format for exchanging data between applications.

```json
{
  "name": "Waleed",
  "experience_years": 2,
  "is_active": true,
  "skills": ["Python", "LLMs", "FastAPI"],
  "location": null
}
```

### Common JSON types

| Type | Example |
| --- | --- |
| String | `"Python"` |
| Number | `25`, `3.14` |
| Boolean | `true`, `false` |
| Array | `["Python", "SQL"]` |
| Object | `{"name": "Waleed"}` |
| Null | `null` |

### JSON rules

- Strings use **double quotes**
- Keys must be **strings**
- **No comments**
- **No trailing commas**
- Use `true`, `false`, `null` — **not** Python's `True`, `False`, `None`

In [ ]:
# Valid vs invalid JSON examples
valid_json = '''
{
    "name": "Waleed",
    "experience_years": 2,
    "skills": ["Python", "LLMs"]
}
'''

invalid_json_trailing_comma = '''
{
    "name": "Waleed",
}
'''

invalid_json_single_quotes = "{'name': 'Waleed'}"

for label, txt in [
    ('Valid', valid_json),
    ('Trailing comma', invalid_json_trailing_comma),
    ('Single quotes', invalid_json_single_quotes),
]:
    try:
        json.loads(txt)
        print(f'✅ {label:16s} → parsed OK')
    except json.JSONDecodeError as e:
        print(f'❌ {label:16s} → JSONDecodeError: {e.msg}')

---
## 📐 3. What Is a Schema?

A **schema** defines the **expected structure of data**.

### Example — support ticket schema

- `issue`: string
- `category`: one of a predefined set
- `priority`: low / medium / high
- `confidence`: number between 0 and 1

Schemas help your application understand what it can **safely expect**.

---
## ⚖️ 4. Prompting for JSON vs Structured Outputs

| Approach | How it works | Main limitation |
| --- | --- | --- |
| **JSON prompting** | Ask model to return JSON | May return invalid/unexpected output |
| **JSON mode** | Provider ensures JSON-formatted output | May not enforce your schema |
| **Schema-constrained output** | Provider constrains to a supported schema | Provider/model support varies |
| **Application validation** | Your code checks the data | Cannot prove facts are true |

> For production, use **provider-enforced structured output** where possible, **then validate** in your application.

---
## 🛡️ 5. Introduction to Pydantic

**Pydantic** validates data against defined models.

### Install

```bash
pip install pydantic pytest
```

### Example model

```python
from typing import Literal
from pydantic import BaseModel, Field

class SupportTicket(BaseModel):
    issue: str = Field(min_length=3)
    category: Literal['account_access', 'billing', 'technical', 'other']
    priority: Literal['low', 'medium', 'high']
    confidence: float = Field(ge=0, le=1)
    needs_human_review: bool
```

### Constraints

- `issue` → at least 3 characters
- `category` → one of the allowed values
- `priority` → low / medium / high
- `confidence` → 0 to 1
- `needs_human_review` → boolean

In [ ]:
# Define the SupportTicket model
if HAS_PYDANTIC:
    from pydantic import BaseModel, Field, ValidationError
    from typing import Literal

    class SupportTicket(BaseModel):
        issue: str = Field(min_length=3)
        category: Literal['account_access', 'billing', 'technical', 'other']
        priority: Literal['low', 'medium', 'high']
        confidence: float = Field(ge=0, le=1)
        needs_human_review: bool

    print('✅ SupportTicket model defined.')
    print('Fields:', list(SupportTicket.model_fields.keys()))
else:
    print('Skipping — install pydantic first.')

---
## 🔍 6. Validate JSON with Pydantic

### What happens during validation

1. Application receives **JSON text**
2. Pydantic **parses** the JSON
3. Pydantic **checks fields & constraints**
4. If valid → **typed Python object**
5. If invalid → **`ValidationError`**

In [ ]:
if HAS_PYDANTIC:
    raw_json = '''
    {
      "issue": "Cannot access my account",
      "category": "account_access",
      "priority": "high",
      "confidence": 0.94,
      "needs_human_review": true
    }
    '''

    try:
        ticket = SupportTicket.model_validate_json(raw_json)
        print('✅ Validated ticket:')
        print(ticket.model_dump())
    except ValidationError as e:
        print('❌ Validation failed:')
        print(e)

In [ ]:
# Validate an existing Python dictionary
if HAS_PYDANTIC:
    data = {
        'issue': 'Payment failed',
        'category': 'billing',
        'priority': 'medium',
        'confidence': 0.85,
        'needs_human_review': False,
    }
    ticket = SupportTicket.model_validate(data)
    print('Category:', ticket.category)
    print('Priority:', ticket.priority)
    print()
    print('As dict:', ticket.model_dump())
    print()
    print('As JSON:')
    print(ticket.model_dump_json(indent=2))

In [ ]:
# Trigger various validation failures
if HAS_PYDANTIC:
    bad_cases = [
        ('issue too short',         {'issue': 'x', 'category': 'billing', 'priority': 'low', 'confidence': 0.5, 'needs_human_review': False}),
        ('invalid category',        {'issue': 'Some issue', 'category': 'urgent', 'priority': 'low', 'confidence': 0.5, 'needs_human_review': False}),
        ('invalid priority',        {'issue': 'Some issue', 'category': 'billing', 'priority': 'urgent', 'confidence': 0.5, 'needs_human_review': False}),
        ('confidence out of range', {'issue': 'Some issue', 'category': 'billing', 'priority': 'low', 'confidence': 1.5, 'needs_human_review': False}),
    ]

    for name, payload in bad_cases:
        try:
            SupportTicket.model_validate(payload)
            print(f'⚠️  {name:22s} → unexpectedly PASSED')
        except ValidationError as e:
            first_err = e.errors()[0]
            print(f'✅ {name:22s} → correctly rejected ({first_err["type"]})')

---
## 🏗️ 7. Build a Structured LLM Extraction Pipeline

Now we combine **LLM APIs**, **prompt engineering**, **JSON**, and **validation** into one pipeline.

### Steps

1. **Define the output model** (Pydantic)
2. **Prepare the input** (customer message)
3. **Write the prompt**
4. **Obtain the raw response**
5. **Parse & validate**
6. **Add business validation**

In [ ]:
# Step 2 & 3 — input + prompt
customer_message = '''
I have been charged twice for my subscription.
Please fix this as soon as possible.
'''

def build_prompt(customer_message: str) -> str:
    return f'''
Analyze the customer message and extract a support ticket.

Allowed categories:
- account_access
- billing
- technical
- other

Allowed priorities:
- low
- medium
- high

Return data matching the required support-ticket structure.
Do not invent facts that are not present in the message.
Set needs_human_review to true if the issue needs manual attention.

Customer message:
<customer_message>
{customer_message}
</customer_message>
'''.strip()

print(build_prompt(customer_message))

In [ ]:
# Step 4 & 5 — parse and validate with a placeholder for the model call
if HAS_PYDANTIC:
    def parse_support_ticket(raw_output: str) -> SupportTicket:
        try:
            data = json.loads(raw_output)
            return SupportTicket.model_validate(data)
        except json.JSONDecodeError as error:
            raise ValueError('The model returned invalid JSON.') from error
        except ValidationError as error:
            raise ValueError('The model output failed schema validation.') from error

    # Example raw output (mocked)
    raw_output = '''
    {
      "issue": "Customer was charged twice",
      "category": "billing",
      "priority": "high",
      "confidence": 0.96,
      "needs_human_review": true
    }
    '''

    ticket = parse_support_ticket(raw_output)
    print('✅ Parsed & validated:')
    print(ticket.model_dump())

In [ ]:
# Step 6 — business rules
if HAS_PYDANTIC:
    def check_business_rules(ticket: SupportTicket) -> None:
        if ticket.confidence < 0.5 and not ticket.needs_human_review:
            raise ValueError(
                'Low-confidence tickets require human review.'
            )

    try:
        check_business_rules(ticket)
        print('✅ Business rules passed.')
    except ValueError as e:
        print('❌ Business rule violation:', e)

    # Test a failing case
    bad_ticket = SupportTicket(
        issue='Test', category='billing', priority='low',
        confidence=0.3, needs_human_review=False,
    )
    try:
        check_business_rules(bad_ticket)
        print('⚠️  Expected a business rule failure.')
    except ValueError as e:
        print('✅ Correctly caught:', e)

---
## 🔒 8. Provider-Enforced Structured Output

Some providers support **structured output directly** through their SDKs.

```python
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.parse(
    model='YOUR_SUPPORTED_MODEL',
    messages=[...],
    response_format=SupportTicket,
)

ticket = response.choices[0].message.parsed
```

### ⚠️ Before running this

1. Install & configure the provider SDK
2. Set `OPENAI_API_KEY`
3. Replace `YOUR_SUPPORTED_MODEL` with a compatible model
4. Check provider docs for API compatibility, refusals, incomplete responses

> Provider-enforced structure **reduces formatting errors** but **does not guarantee factual correctness** or **business rule compliance**.

In [ ]:
# Run a real provider-enforced call only if credentials exist
if HAS_KEY and HAS_MODEL and HAS_PYDANTIC:
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
        load_dotenv()

        client = OpenAI(
            api_key=os.environ['LLM_API_KEY'],
            base_url=os.getenv('LLM_BASE_URL') or None,
        )
        response = client.chat.completions.parse(
            model=os.environ['LLM_MODEL'],
            messages=[
                {'role': 'system', 'content': 'Extract a support ticket. Do not invent missing facts.'},
                {'role': 'user',   'content': 'I was charged twice for my subscription.'},
            ],
            response_format=SupportTicket,
        )
        parsed = response.choices[0].message.parsed
        if parsed is None:
            print('No parsed content returned.')
        else:
            print('✅ Provider-parsed ticket:')
            print(parsed.model_dump())
    except Exception as e:
        print(f'Live call failed: {type(e).__name__}: {e}')
else:
    print('Skipped live provider call (no credentials or provider not configured).')

---
## 🚨 9. Error Handling & Recovery

| Failure | Example | Response |
| --- | --- | --- |
| Invalid JSON | Missing brace | Reject / bounded repair |
| Missing field | No `priority` | Reject / request corrected |
| Invalid enum | `"urgent"` | Reject / explicit mapping |
| Wrong type | `confidence: "high"` | Reject / documented conversion |
| Invalid business value | Confidence out of range | Reject |
| Refusal / incomplete | Model declines | Handle separately |

### Safe recovery strategy

1. Try to **parse** the response
2. **Validate** against the schema
3. **Record** the failure safely
4. Optionally attempt a **limited** correction
5. **Validate** the corrected result
6. Stop after a small, configured number of attempts
7. Send unresolved cases to **fallback / human review**

> ⚠️ **Never retry indefinitely.** **Never** silently fabricate values.

---
## 🔀 10. Structured Outputs vs Tool Calling

| Structured Outputs | Tool Calling |
| --- | --- |
| Produces data in expected format | Requests an operation |
| Example: extract invoice total | Example: request invoice lookup |
| Output parsed & validated | Tool args validated before execution |
| Doesn't itself execute actions | Application decides whether to execute |

> A model might return a **structured request** to look up an invoice. Your backend must still check **permissions**, **validate the ID**, and **authorize** before making the lookup.

→ We'll cover **tool calling** in **Module 11**.

---
## 🛡️ 11. Reliability, Security & Best Practices

- **Validate every response** — never assume correctness
- **Use explicit schemas** — required fields, allowed values, boundaries
- **Separate validation layers** — parse → schema → business rules → fact check
- **Treat output as untrusted** — never execute model code / SQL directly
- **Protect sensitive data** — avoid sending secrets / PII
- **Authorize actions in app code** — the model cannot grant permissions
- **Handle refusals & incomplete output** — not every response is a successful extraction
- **Keep logs safe** — no credentials, keys, or sensitive content
- **Test edge cases** — empty input, unusual chars, conflicting instructions
- **Measure quality** — schema-validity rate vs factual accuracy vs task success

> 🧭 **A schema constrains the shape of a response — it cannot prove the response is true.**

---
## 🏋️ 12. Practical Lab — Support Ticket Extractor

### 🎯 Objective
Build a reusable Python app that converts customer messages into **validated support-ticket objects**.

### Suggested structure

```text
05-Structured-Outputs-and-Validation/
├── README.md
├── models.py
├── prompts.py
├── parser.py
├── validator.py
├── main.py
├── tests/
│   ├── test_models.py
│   ├── test_parser.py
│   └── test_business_rules.py
└── data/
    └── sample_tickets.jsonl
```

### Requirements

1. Pydantic model for support tickets
2. Prompt template for extraction
3. LLM provider connection
4. Parse & validate the response
5. Business rules for low-confidence outputs
6. Handle malformed JSON & invalid fields
7. Automated tests
8. Small dataset of sample messages + expected outputs
9. Report validation success & extraction accuracy
10. Document limitations & failure handling

### Required test cases

- Normal billing complaint
- Password-reset request
- Message with no useful info
- Output missing a required field
- Invalid category / priority
- Confidence outside allowed range
- Malformed JSON
- Message trying to override the extraction task

> Use **mocked LLM responses** for deterministic tests. Keep separate **evaluation tests** for real models.

In [ ]:
# A runnable scaffold for the parser + business rules
if HAS_PYDANTIC:
    from pydantic import BaseModel, Field, ValidationError
    from typing import Literal

    class Ticket(BaseModel):
        issue: str = Field(min_length=3)
        category: Literal['account_access', 'billing', 'technical', 'other']
        priority: Literal['low', 'medium', 'high']
        confidence: float = Field(ge=0, le=1)
        needs_human_review: bool

    def parse_and_validate(raw: str) -> Ticket:
        try:
            data = json.loads(raw)
            return Ticket.model_validate(data)
        except json.JSONDecodeError as e:
            raise ValueError(f'Invalid JSON: {e.msg}') from e
        except ValidationError as e:
            raise ValueError(f'Schema validation failed: {e.errors()[0]["type"]}') from e

    # Mocked model outputs
test_outputs = {
    'valid':            '{"issue": "Cannot log in", "category": "account_access", "priority": "high",   "confidence": 0.9,  "needs_human_review": true}',
    'missing field':    '{"issue": "Cannot log in", "category": "account_access", "priority": "high",   "confidence": 0.9}',
    'bad category':     '{"issue": "Cannot log in", "category": "urgent",         "priority": "high",   "confidence": 0.9,  "needs_human_review": true}',
    'out of range':     '{"issue": "Cannot log in", "category": "account_access", "priority": "high",   "confidence": 1.5,  "needs_human_review": true}',
    'bad json':         '{"issue": "Cannot log in", "category": "account_access", "priority": "high",   "confidence": 0.9,',
}

for name, raw in test_outputs.items():
    try:
        ticket = parse_and_validate(raw)
        print(f'✅ {name:16s} → {ticket.model_dump()}')
    except ValueError as e:
        print(f'❌ {name:16s} → {e}')

---
## 🎤 13. Interview Questions

**Q1. What are structured outputs in GenAI?**  
Responses that follow an expected data format — e.g., JSON matching a defined schema.

**Q2. Why is prompting for JSON not enough?**  
The model may return invalid JSON, missing fields, unexpected values, or wrong facts. Applications need validation & error handling.

**Q3. What is Pydantic?**  
A Python library that validates data using typed models and field constraints.

**Q4. JSON parsing vs schema validation?**  
Parsing checks if text is valid JSON. Schema validation checks if the parsed data matches the required structure & constraints.

**Q5. Does valid structured output guarantee factual accuracy?**  
No. A response can match every schema rule and still be factually wrong.

**Q6. JSON mode vs schema-constrained output?**  
JSON mode focuses on valid JSON formatting. Schema-constrained output also enforces structure — depends on provider/model support.

**Q7. How should an app handle invalid model output?**  
Reject, log safely, optionally attempt limited correction, validate again, fall back / human review when needed.

**Q8. Why validate model output before DB operations?**  
Validation reduces malformed data & enforces rules. Auth & transaction checks still required before sensitive ops.

**Q9. How would you test an LLM extraction pipeline?**  
Deterministic unit tests with mocked outputs, edge-case tests, and a separate evaluation dataset for extraction quality.

**Q10. Structured outputs vs tool calling?**  
Structured outputs return data in a defined format. Tool calling requests an operation that the app may execute after validation & authorization.

---
## ✅ 14. Module Completion Checklist

- [ ] Understand JSON syntax & data types
- [ ] Understand schemas & required fields
- [ ] Compare JSON prompting, JSON mode, schema-constrained output
- [ ] Create Pydantic models with field constraints
- [ ] Validate JSON strings & Python dictionaries
- [ ] Handle parsing & validation errors
- [ ] Add business-rule validation
- [ ] Integrate a supported provider's structured-output feature
- [ ] Write unit tests using mocked responses
- [ ] Evaluate extraction accuracy with sample data
- [ ] Complete the Support Ticket Extractor project

---
## 🏁 15. Key Takeaways

- LLMs generate content, but applications need **dependable data contracts**
- **JSON** is a data format; a **schema** describes the expected structure
- **Pydantic** helps validate & work with **typed Python data**
- **Provider-enforced structured output** can improve formatting reliability
- **Schema validity ≠ factual accuracy**
- **Business rules, authorization, testing, error handling** remain application responsibilities
- Production GenAI engineering requires **measuring reliability**, not just generating a successful response

### 🧭 Golden Rule

> **A schema constrains the shape of a response — it cannot prove the response is true.**

---

### 🔗 What's Next?

**Module 06 — Embeddings and Semantic Search**

- Represent text as **numerical vectors**
- Compare meaning **beyond exact keyword matching**
- **Cosine similarity**, **vector search**
- Practical use cases: **retrieval**, **recommendation**, **clustering**